# Factor Analysis and Multiple Regression

**Course alignment:** Advanced Statistical Methods  
**Type:** Original supplementary practice  
**Scenario:** Analysing service quality and operational drivers in a fictional electricity provider

This notebook demonstrates exploratory factor analysis on simulated survey items and multiple linear regression with diagnostics. It distinguishes latent constructs from observed variables and association from causation. All data are synthetic.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

rng=np.random.default_rng(303)
n=1200
# Two latent constructs: service quality and supply reliability.
latent=rng.multivariate_normal(
    mean=[0,0],cov=[[1,.35],[.35,1]],size=n
)
service,reliability=latent.T
items=pd.DataFrame({
    "response_speed":3.2+.75*service+rng.normal(0,.45,n),
    "staff_helpfulness":3.3+.80*service+rng.normal(0,.45,n),
    "complaint_resolution":3.0+.70*service+.12*reliability+rng.normal(0,.5,n),
    "outage_frequency":3.1-.80*reliability+rng.normal(0,.45,n),
    "voltage_stability":3.2+.85*reliability+rng.normal(0,.45,n),
    "restoration_speed":3.0+.65*reliability+.1*service+rng.normal(0,.5,n)
})
display(items.describe().round(2))
print("Correlation matrix")
display(items.corr().round(2))

## 1. Factor analysis

Factor analysis models shared variation among observed measures using a smaller number of latent factors. Standardise item scales before fitting. Factor direction and ordering are arbitrary; interpret the loading pattern rather than the sign alone.

In [ ]:
scaler=StandardScaler()
Z=scaler.fit_transform(items)
fa=FactorAnalysis(n_components=2,random_state=42)
factor_scores=fa.fit_transform(Z)
loadings=pd.DataFrame(
    fa.components_.T,
    index=items.columns,
    columns=["Factor_1","Factor_2"]
)
display(loadings.round(3))

fig,ax=plt.subplots(figsize=(8,4.5))
loadings.plot(kind="bar",ax=ax)
ax.set(title="Exploratory factor loadings",ylabel="Loading",xlabel="Observed survey item")
ax.axhline(0,color="black",linewidth=.8)
plt.xticks(rotation=35,ha="right")
plt.tight_layout()
plt.show()

## 2. Create a factor-score dataset

The factor scores are estimated from observed indicators. They are not error-free measurements of the latent constructs and should not be treated as directly observed ground truth.

In [ ]:
score_df=pd.DataFrame(factor_scores,columns=["factor_1_score","factor_2_score"])
score_df.describe().round(2)

## 3. Multiple linear regression

Now create an operational dataset to estimate associations with a synthetic satisfaction score. Predictors include service delay, outage duration, bill amount and household size. The target is bounded to a 1–5 scale for realism, but OLS is used as a teaching baseline; ordinal regression could be a more suitable extension.

In [ ]:
service_delay=rng.gamma(shape=2.2,scale=1.4,size=n)
outage_hours=rng.gamma(shape=1.7,scale=2.0,size=n)
bill_amount=rng.normal(1800,450,n).clip(300,4000)
household_size=rng.integers(1,8,n)
latent_sat=4.7-.18*service_delay-.14*outage_hours-.00012*bill_amount+.04*household_size+rng.normal(0,.35,n)
satisfaction=np.clip(latent_sat,1,5)

reg=pd.DataFrame({
    "satisfaction":satisfaction,
    "service_delay_hours":service_delay,
    "outage_hours":outage_hours,
    "bill_amount":bill_amount,
    "household_size":household_size
})
ols=smf.ols(
    "satisfaction ~ service_delay_hours + outage_hours + bill_amount + household_size",
    data=reg
).fit()
print(ols.summary())

## 4. Standardised coefficients

Standardised coefficients put predictors on a common standard-deviation scale. They can help compare conditional associations, but they do not resolve confounding or establish causal importance.

In [ ]:
X=reg[["service_delay_hours","outage_hours","bill_amount","household_size"]]
Xz=pd.DataFrame(StandardScaler().fit_transform(X),columns=X.columns)
yz=(reg["satisfaction"]-reg["satisfaction"].mean())/reg["satisfaction"].std()
standardised=sm.OLS(yz,sm.add_constant(Xz)).fit()
display(standardised.params.drop("const").sort_values().to_frame("standardised_beta"))

## 5. Multicollinearity and residual diagnostics

In [ ]:
X_design=sm.add_constant(reg[["service_delay_hours","outage_hours","bill_amount","household_size"]])
vif=pd.DataFrame({
    "feature":X_design.columns,
    "VIF":[variance_inflation_factor(X_design.values,i) for i in range(X_design.shape[1])]
})
display(vif.round(2))

fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].scatter(ols.fittedvalues,ols.resid,alpha=.3,s=12)
ax[0].axhline(0,linestyle="--")
ax[0].set(title="Residuals vs fitted",xlabel="Fitted satisfaction",ylabel="Residual")
sm.qqplot(ols.resid,line="45",ax=ax[1],fit=True)
ax[1].set_title("Residual Q-Q plot")
plt.tight_layout()
plt.show()

print("Breusch-Pagan test:",sm.stats.het_breuschpagan(ols.resid,ols.model.exog))

## Interpretation tasks

1. Which items tend to load together? Does the pattern resemble the two simulated constructs?
2. Why can factor signs flip without changing the underlying factor model?
3. Interpret each multiple regression coefficient while holding other predictors constant.
4. Compare raw and standardised coefficients. Why are they not causal effects?
5. What do VIF and residual plots suggest about model adequacy?
6. Explain why an ordinal model may be preferable for a 1–5 satisfaction response.

**Limitations:** The data-generating process is known and synthetic. Factor analysis is exploratory, factor rotations are not implemented here, and OLS is used as a teaching baseline.